神经网络由对数据执行运算的层/模块构成。torch.nn命名空间提供了搭建神经网络所需的全部组件。

PyTorch 中每一个模块都继承自nn.Module。神经网络本身也是一个模块，它内部又包含其他子模块（各个网络层）。这种嵌套结构让搭建、管理复杂网络架构变得十分便捷。

接下来，我们将搭建一个神经网络，对 FashionMNIST 数据集的图片做分类。

**整理者会为这部分以及之后的部分写一个py文件，让各位能整体了解神经网络搭建的流程**

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

### PART1.获取训练设备

我们希望可以在硬件加速器上训练模型，例如 CUDA、MPS、MTIA、XPU。如果当前加速器可用，就使用它；否则就使用 CPU。

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"使用设备： {device} ")

使用设备： cpu 


### PART2.定义模型类

我们通过继承nn.Module来定义神经网络，在__init__里面初始化网络各层。
所有nn.Module的子类，都要在forward 方法中实现对输入数据的运算逻辑。

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__() #调用父类构造函数，初始化父类所有内容
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

我们创建NeuralNetwork的实例，迁移到指定设备，打印网络结构。

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


使用模型时，我们直接把输入数据传给模型对象。这会执行模型的forward前向函数，同时附带一些后台内部操作。不要直接调用model.forward()！

将输入传入模型，会返回一个二维张量：
第 0 维对应样本，每个样本输出 10 个原始预测值（对应 10 个类别）；第 1 维对应每个类别的分数。

把输出送入nn.Softmax模块，就可以得到各类别的预测概率。

In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"预测类别: {y_pred}")

预测类别: tensor([3])


### PART3.模型各层

我们来拆解 FashionMNIST 模型里面每一层。为方便演示，取一个小批次，包含 3 张 28×28 的图片，观察数据流过网络每一层时发生的变化。

In [6]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


**nn.Flatten**

我们实例化nn.Flatten层，把每张 2 维 28×28 图片，转换成由 784 个像素值组成的一维连续数组；批次维度 dim=0 会被保留不变。

In [7]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


**nn.Linear**

线性层是一个模块，利用内部存储的权重weight和偏置bias，对输入做线性变换。

In [8]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
#等价于hidden1 = nn.Linear(in_features=28*28, out_features=20)(flat_image)
print(hidden1.size())

torch.Size([3, 20])


**nn.ReLU**

非线性激活函数，用来在模型输入和输出之间构建复杂映射关系。
一般放在线性变换之后，引入非线性，让神经网络有能力学习各种各样复杂的数据规律。

在这个模型中，我们在线性层之间使用nn.ReLU；当然还有其他激活函数，同样可以给网络引入非线性。

In [9]:
print(f"使用ReLU前: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"使用ReLU后: {hidden1}")

使用ReLU前: tensor([[ 0.0330,  0.0889, -0.0337, -0.1201, -0.1280,  0.1336, -0.4768, -0.0020,
         -0.0407, -0.3179,  0.3264,  0.2645,  0.0795,  0.4164, -0.4233,  0.2643,
          0.2062,  0.2415,  0.3771, -0.2745],
        [ 0.1680,  0.0601, -0.2332, -0.0237, -0.0575,  0.2970, -0.1060,  0.0625,
         -0.4896, -0.2282,  0.0565,  0.2557, -0.0868,  0.5228, -0.4617, -0.0575,
          0.0432, -0.1626,  0.3944, -0.0454],
        [ 0.0275,  0.1137, -0.2935, -0.0679,  0.2844,  0.1219, -0.5590, -0.1556,
         -0.1905, -0.2857,  0.1946,  0.3036,  0.1900,  0.2592, -0.4627,  0.2675,
          0.1226, -0.0844, -0.0439, -0.4903]], grad_fn=<AddmmBackward0>)


使用ReLU后: tensor([[0.0330, 0.0889, 0.0000, 0.0000, 0.0000, 0.1336, 0.0000, 0.0000, 0.0000,
         0.0000, 0.3264, 0.2645, 0.0795, 0.4164, 0.0000, 0.2643, 0.2062, 0.2415,
         0.3771, 0.0000],
        [0.1680, 0.0601, 0.0000, 0.0000, 0.0000, 0.2970, 0.0000, 0.0625, 0.0000,
         0.0000, 0.0565, 0.2557, 0.0000, 0.5228, 0.0000, 0.0

**nn.Sequential**

nn.Sequential是有序模块容器。输入数据会严格按照定义顺序依次经过内部每一个模块。你可以用 Sequential 快速拼接搭建网络，示例如下：

In [10]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

**nn.Softmax**

神经网络最后一层线性层输出logits（原始分数，取值范围 (-infty,+infty)），将 logits 送入nn.Softmax模块。

logits 会被映射到区间[0,1]，得到模型对各个类别的预测概率。

dim参数指定：沿着哪一个维度做运算，该维度上所有数值求和等于 1。

In [11]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

### PART4.模型参数

神经网络内部很多层都是带参数的：也就是存有权重 weight和偏置 bias，这些数值会在训练过程中被优化更新。

继承nn.Module之后，框架会自动追踪模型对象内部定义的所有模块；可以通过模型的parameters()或者named_parameters()获取全部参数。

下面示例遍历每一个参数，打印参数形状，同时截取一小部分数值预览。

In [12]:
# 打印完整网络结构
print(f"模型结构: {model}\n\n")

# named_parameters()：同时返回【参数名字】和【参数张量】
for name, param in model.named_parameters():
    # name：参数名字，例如 linear_relu_stack.0.weight
    # param.size()：该参数张量的shape
    # param[:2]：截取前2个元素，用来简单看一下参数数值
    print(f"层: {name} | 形状: {param.size()} | 数值预览 : {param[:2]} \n")

模型结构: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


层: linear_relu_stack.0.weight | 形状: torch.Size([512, 784]) | 数值预览 : tensor([[ 0.0214, -0.0320,  0.0110,  ..., -0.0060, -0.0190, -0.0007],
        [-0.0014,  0.0225, -0.0060,  ...,  0.0037,  0.0235,  0.0250]],
       grad_fn=<SliceBackward0>) 

层: linear_relu_stack.0.bias | 形状: torch.Size([512]) | 数值预览 : tensor([ 0.0111, -0.0263], grad_fn=<SliceBackward0>) 

层: linear_relu_stack.2.weight | 形状: torch.Size([512, 512]) | 数值预览 : tensor([[ 0.0226,  0.0276,  0.0412,  ..., -0.0165,  0.0091, -0.0282],
        [-0.0306,  0.0411, -0.0264,  ..., -0.0330, -0.0251,  0.0251]],
       grad_fn=<SliceBackward0>) 

层: linear_relu_stack.2.bias | 形状: torch.Size([512]) | 数值预览 : tensor([